In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# --- 1. SCRAPING ---
base_url = "http://books.toscrape.com/catalogue/"

# We scrape 3 specific categories to easily capture the 'category' field and exceed 60 books.
category_urls = [
    ("Travel", "category/books/travel_2/index.html"),
    ("Mystery", "category/books/mystery_3/index.html"),
    ("Historical Fiction", "category/books/historical-fiction_4/index.html")
]

book_data = []

for cat_name, cat_url in category_urls:
    current_url = base_url + cat_url
    while current_url:
        response = requests.get(current_url)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        books = soup.find_all('article', class_='product_pod')
        for book in books:
            book_data.append({
                'title': book.h3.a['title'],
                'price': book.find('p', class_='price_color').text,
                'star_rating': book.p['class'][1], 
                'availability': book.find('p', class_='instock availability').text.strip(),
                'category': cat_name
            })
            
        # Pagination logic
        next_btn = soup.find('li', class_='next')
        if next_btn:
            next_url = next_btn.a['href']
            current_url = current_url.rsplit('/', 1)[0] + '/' + next_url
        else:
            current_url = None

df = pd.DataFrame(book_data)
print(f"Total books scraped: {len(df)}") # Should be 69 books

# --- 2. CLEANING & CONVERSION ---
# Strip currency symbol and encoding artifacts, then convert to float
df['price_gbp'] = df['price'].str.replace('£', '', regex=False) \
                             .str.replace('Â£', '', regex=False) \
                             .str.replace('Â', '', regex=False) \
                             .astype(float)

# Convert text ratings to integers
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['star_rating'].map(rating_map)

# Parse availability to boolean
df['in_stock'] = df['availability'].str.contains('In stock', case=False, na=False)

# Convert to INR using the REQUIRED fixed baseline rate
FIXED_CONVERSION_RATE = 105.50
df['price_inr'] = df['price_gbp'] * FIXED_CONVERSION_RATE

# Drop the old uncleaned columns
df = df.drop(columns=['price', 'star_rating', 'availability'])

# Handle any messy rows (Rubric requirement)
if df.isnull().sum().sum() > 0:
    df = df.dropna()

display(df.head())

Total books scraped: 69


,title,category,price_gbp,rating,in_stock,price_inr
0,It's Only the Himalayas,Travel,45.17,2,True,4765.435
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Travel,49.43,4,True,5214.865
2,See America: A Celebration of Our National Par...,Travel,48.87,3,True,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,True,3897.170
4,Under the Tuscan Sun,Travel,37.33,3,True,3938.315


In [3]:
df['category'].value_counts()

category
Mystery               32
Historical Fiction    26
Travel                11
Name: count, dtype: int64

In [ ]:
import sqlite3

# --- 3. DATABASE SCHEMA & LOADING ---
# Create a normalized schema: separate 'categories' and 'books' tables
categories_df = pd.DataFrame({'category_name': df['category'].unique()})
categories_df['category_id'] = range(1, len(categories_df) + 1)

# Merge category_id back to books and drop the text category column
books_df = df.merge(categories_df, left_on='category', right_on='category_name')
books_df = books_df.drop(columns=['category', 'category_name'])
books_df['book_id'] = range(1, len(books_df) + 1)

# Reorder columns for neatness
books_df = books_df[['book_id', 'title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]

# Connect to SQLite and load data
conn = sqlite3.connect('zepto_catalog.sqlite')
categories_df.to_sql('categories', conn, if_exists='replace', index=False)
books_df.to_sql('books', conn, if_exists='replace', index=False)

# --- 4. SQL QUERIES ---
queries = {
    "1. DISTINCT": "SELECT DISTINCT category_name FROM categories;",
    "2. WHERE & BETWEEN": "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 30;",
    "3. IN & ORDER BY": "SELECT title, rating FROM books WHERE category_id IN (1, 2) ORDER BY rating DESC LIMIT 5;",
    "4. LIMIT": "SELECT title, in_stock FROM books LIMIT 5;",
    "5. JOIN (Highest rated books with their category names)": """
        SELECT c.category_name, b.title, b.rating 
        FROM books b 
        JOIN categories c ON b.category_id = c.category_id 
        WHERE b.rating = 5 
        ORDER BY b.title 
        LIMIT 5;
    """
}

print("--- SQL EXECUTIONS ---")
for name, query in queries.items():
    print(f"\n{name}\nQuery: {query}")
    display(pd.read_sql(query, conn))

# --- 5. PANDAS VS SQL COMPARISON ---
print("\n--- PANDAS VS SQL JOIN COMPARISON ---")
# 1. Read the JOIN query using read_sql (Requirement)
sql_join_result = pd.read_sql(queries["5. JOIN (Highest rated books with their category names)"], conn)

# 2. Reproduce the exact same result using pandas.merge (Requirement)
pandas_join = pd.merge(books_df, categories_df, on='category_id')
pandas_join_result = pandas_join[pandas_join['rating'] == 5][['category_name', 'title', 'rating']]
pandas_join_result = pandas_join_result.sort_values(by='title').head(5).reset_index(drop=True)

print("Result from pd.read_sql:")
display(sql_join_result)
print("\nResult from pd.merge:")
display(pandas_join_result)

# Verify they match
print("Do the results match exactly?", sql_join_result.equals(pandas_join_result))

conn.close()

--- SQL EXECUTIONS ---

1. DISTINCT
Query: SELECT DISTINCT category_name FROM categories;


,category_name
0,Travel
1,Mystery
2,Historical Fiction



2. WHERE & BETWEEN
Query: SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20 AND 30;


,title,price_gbp
0,The Road to Little Dribbling: Adventures of an...,23.21
1,"1,000 Places to See Before You Die",26.08
2,Poisonous (Max Revere Novels #3),26.80
3,The Widow,27.26
4,What Happened on Beale Street (Secrets of the ...,25.37
5,Delivering the Truth (Quaker Midwife Mystery #1),20.89
6,The Mysterious Affair at Styles (Hercule Poiro...,24.80
7,The Silkworm (Cormoran Strike #2),23.05
8,Extreme Prey (Lucas Davenport #26),25.40
9,Career of Evil (Cormoran Strike #3),24.72



3. IN & ORDER BY
Query: SELECT title, rating FROM books WHERE category_id IN (1, 2) ORDER BY rating DESC LIMIT 5;


...

,category_name,title,rating
0,Travel,"1,000 Places to See Before You Die",5
1,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5
2,Historical Fiction,A Spy's Devotion (The Regency Spies of London #1),5
3,Mystery,A Time of Torment (Charlie Parker #14),5
4,Historical Fiction,Between Shades of Gray,5



Result from pd.merge:


,category_name,title,rating
0,Travel,"1,000 Places to See Before You Die",5
1,Historical Fiction,A Flight of Arrows (The Pathfinders #2),5
2,Historical Fiction,A Spy's Devotion (The Regency Spies of London #1),5
3,Mystery,A Time of Torment (Charlie Parker #14),5
4,Historical Fiction,Between Shades of Gray,5


Do the results match exactly? True


: 